# BT4012 — Fraudulent Bitcoin Transaction Detection

**Final model:** ten-seed ordered CatBoost, depth 4, 1,500 trees at learning
rate 0.03, `l2_leaf_reg` 30, 23 feature borders.

**Run from the project root.** Install the packages pinned in `requirements.txt`.


## 1. Setup

In [1]:
import hashlib
import platform
from pathlib import Path
import time

import numpy as np
import pandas as pd
import catboost
from catboost import CatBoostClassifier
from joblib import Parallel, delayed
from scipy.stats import rankdata
import sklearn

ROOT = Path.cwd()
DATA = ROOT / 'data'
OUT = ROOT / 'notebook_output'
OUT.mkdir(exist_ok=True)
SUBMITTED = ROOT / 'submissions/submission_20261023_catboost_ordered_iter1500_border23.csv'

print('Python', platform.python_version())
for m in (np, pd, catboost, sklearn):
    print(f'{m.__name__:10s}', m.__version__)

Python 3.13.5
numpy      2.5.3
pandas     3.0.5
catboost   1.2.10
sklearn    1.9.0


## 2. Data

Only labelled training rows are used: 31,235 rows from time steps 1–35, 11.67%
of them illicit. The test set is 15,329 rows from time steps 36–49. All 165
anonymised features are used, both local and neighbour-aggregated. No
engineered features are added: graph, autoencoder, kNN and PCA features all
lost on the public leaderboard in earlier experiments.

In [2]:
train = pd.read_csv(DATA / 'train.csv').dropna(subset=['label']).reset_index(drop=True)
test = pd.read_csv(DATA / 'test.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')
FEATURES = [f'feat_{i}' for i in range(1, 166)]

assert len(train) == 31235 and len(test) == 15329
assert np.array_equal(test['index'], sample['index'])
assert train.time_step.max() == 35 and test.time_step.min() == 36
print(f'train: {len(train):,} labelled rows, illicit rate {train.label.mean():.4f}')
print(f'test:  {len(test):,} rows, time steps {test.time_step.min()}-{test.time_step.max()}')

train: 31,235 labelled rows, illicit rate 0.1167
test:  15,329 rows, time steps 36-49


## 3. Preprocessing

Test features are clipped to the training minimum and maximum, so the trees never
see values outside the training range. Both sets are then standardised with the
training mean and population standard deviation. Standardisation is kept from
the earlier linear-leaf model; it does not change tree splits materially.

In [3]:
def prep(train, test, features):
    x = train[features]
    xt = test[features].clip(x.min(), x.max(), axis=1)
    mu = x.mean()
    sd = x.std(ddof=0)
    sd = sd.where(sd > 0, 1.)
    return (x - mu) / sd, (xt - mu) / sd

x, xt = prep(train, test, FEATURES)
y = train.label.astype(int).to_numpy()
print(x.shape, xt.shape)

(31235, 165) (15329, 165)


## 4. Model

CatBoost with **ordered boosting**: each row's gradient comes from a model that
never saw that row, which reduces the prediction shift that ordinary boosting
builds up on its own training rows. Symmetric depth-4 trees, strong leaf
regularisation (`l2_leaf_reg=30`), 1,500 trees and coarse feature binning
(`border_count=23`: each feature is split on at most 23 candidate thresholds)
were each chosen by one-change public-leaderboard experiments. Ten seeds are trained, each with two threads, so
results are deterministic.

In [4]:
PARAMS = dict(depth=4, iterations=1500, learning_rate=0.03, border_count=23, rsm=0.8,
              l2_leaf_reg=30, loss_function='Logloss', boosting_type='Ordered',
              verbose=0, allow_writing_files=False, thread_count=2)
SEEDS = [42, 1, 7, 13, 99, 2024, 5, 77, 123, 31]

def fit_one(seed):
    model = CatBoostClassifier(**PARAMS, random_seed=seed)
    model.fit(x, y)
    return model.predict_proba(xt)[:, 1]

start = time.monotonic()
preds = Parallel(n_jobs=4)(delayed(fit_one)(s) for s in SEEDS)
p = np.array(preds)[None, :, :]   # shape (1 family, 10 seeds, 15,329 rows)
print(f'trained {len(SEEDS)} models in {time.monotonic() - start:.0f}s; shape {p.shape}')

trained 10 models in 596s; shape (1, 10, 15329)


## 5. Pooling and submission

The ten models are combined by averaging two rank-based scores equally:

- each model's percentile rank, averaged across models;
- the percentile rank of the models' mean log-odds.

The result is then percentile-ranked over the whole test batch. ROC-AUC depends
only on ordering, so the output is a rank score in (0, 1].

In [5]:
def rank(v):
    return rankdata(v, method='average') / len(v)

def pool(p):
    ranks = np.array([[rank(q) for q in fam] for fam in p])
    q = np.clip(p, 1e-15, 1 - 1e-15)
    logs = (np.log(q) - np.log1p(-q)).mean(axis=0).mean(axis=0)
    return rank(.5 * ranks.mean(axis=0).mean(axis=0) + .5 * rank(logs))

submission = pd.DataFrame({'index': sample['index'], 'target': rank(pool(p))})
path = OUT / 'submission.csv'
submission.to_csv(path, index=False)

saved = pd.read_csv(path)
assert list(saved) == ['index', 'target'] and len(saved) == 15329
assert np.array_equal(saved['index'], sample['index']) and saved.target.between(0, 1).all()
print(path.relative_to(ROOT), hashlib.sha256(path.read_bytes()).hexdigest())

notebook_output/submission.csv c30d83966fdeb1c7f06c3254648a0682a5890b1094e561aa0f046b7ceecbb54a
